# TP-1: Cleaning Data — Applying OOP and Design Patterns

> **Course**: Advanced Programming / Data Science  
> **Topic**: Object-Oriented Programming (OOP), SOLID Principles & Design Patterns  
> **Goal**: Build a complete, modular, extensible CSV data cleaning pipeline using OOP, decorators, and design patterns.

---

### Structure of This Lab
Each exercise builds progressively upon the previous one:
1. **Exercise 1: OOP Foundation** — Build the `CSVReader` class (encapsulation, constructors, methods).
2. **Exercise 2: Strategy Pattern** — Design interchangeable missing-value handling algorithms (`DropMissing`, `FillMean`, `FillMode`, `DataCleaner`).
3. **Exercise 3: Decorator Pattern** — Implement reusable logging (`@log_action`) and performance profiling (`@log_time`).
4. **Exercise 4: Factory Pattern** — Implement a scalable factory (`TransformFactory`) for dynamic data transformations (`NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`).
5. **Exercise 5: Template Method Pattern** — Compose a complete end-to-end pipeline (`DataPipeline` and `CSVDataPipeline`) orchestrating load, clean, transform, and save.


## 0. Setup and Imports

In [1]:
import os
import time
import functools
from abc import ABC, abstractmethod
from typing import List, Optional, Union

import pandas as pd
import numpy as np

# Print versions for reproducibility
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")


Pandas version: 2.2.2
NumPy version:  1.26.4


---
## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Objective
- Reinforce **OOP fundamentals**: classes, constructors (`__init__`), instance variables, methods, encapsulation.
- Learn to design reusable, self-contained data ingestion classes for Data Science workflows.

### Concept
A `CSVReader` class encapsulates all functionality related to reading and inspecting CSV files. Rather than calling ad-hoc `pd.read_csv()` functions throughout the codebase, encapsulation guarantees consistent data loading, validation, and state management.

### Instructions
1. Create a class `CSVReader`.
2. Define an `__init__` method that accepts `file_path`.
3. Add a `read()` method that loads the CSV file using pandas and stores it in an instance attribute.
4. Add a `preview(n)` method that displays or prints the top `n` rows (default 5).


In [2]:
class CSVReader:
    """
    Encapsulates CSV loading and inspection functionality.
    
    Attributes:
        file_path (str): Path to the target CSV file.
        data (pd.DataFrame, optional): Loaded pandas DataFrame.
    """
    def __init__(self, file_path: str):
        self.file_path: str = file_path
        self.data: Optional[pd.DataFrame] = None

    def read(self) -> pd.DataFrame:
        """
        Loads the CSV file into a pandas DataFrame.
        
        Returns:
            pd.DataFrame: The loaded dataset.
        Raises:
            FileNotFoundError: If the specified file_path does not exist.
        """
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"Error: CSV file not found at '{self.file_path}'")
        
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n: int = 5) -> pd.DataFrame:
        """
        Displays the first n rows of the loaded dataset.
        
        Args:
            n (int): Number of top rows to preview (default is 5).
            
        Returns:
            pd.DataFrame: The top n rows.
        """
        if self.data is None:
            self.read()
        return self.data.head(n)


### Verification — Exercise 1: Testing `CSVReader`

In [3]:
# Instantiate CSVReader with the sample dataset
sample_file = "sample_data (2).csv"
reader = CSVReader(sample_file)

# Read the dataset
df_raw = reader.read()
print(f"Successfully loaded '{sample_file}' with shape: {df_raw.shape}")

# Preview top rows
reader.preview()


Successfully loaded 'sample_data (2).csv' with shape: (7, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


---
## Exercise 2: Apply Pattern for Missing Values Handling (Strategy Pattern)

### Objective
- Master the **Strategy Pattern**: define a family of algorithms, encapsulate each one, and make them interchangeable.
- Practice **inheritance**, **polymorphism**, and abstract base classes (`abc.ABC`).

### Concept
Real-world datasets require diverse missing data strategies depending on the column type and task:
- Dropping rows with missing values (`DropMissing`)
- Imputing with the column mean (`FillMean`) for numerical features
- Imputing with the column mode (`FillMode`) for categorical/discrete features

The **Strategy Pattern** allows the `DataCleaner` client to seamlessly switch imputation techniques at runtime **without modifying existing cleaner code**, fully satisfying the Open/Closed Principle.

### Instructions
1. Create an abstract class `MissingValueStrategy` with an abstract method `handle(df: pd.DataFrame) -> pd.DataFrame`.
2. Implement concrete strategy subclasses:
   - `DropMissing`: drops rows with missing values.
   - `FillMean`: fills numerical columns with their arithmetic mean.
   - `FillMode`: fills columns with their most frequent value (mode).
3. Create a `DataCleaner` context class that receives a strategy and executes it.


In [5]:
class MissingValueStrategy(ABC):
    """Abstract base class (Strategy interface) for handling missing data."""
    
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies a missing value handling algorithm to the DataFrame.
        
        Args:
            df (pd.DataFrame): Input DataFrame with potential missing values.
            
        Returns:
            pd.DataFrame: Cleaned DataFrame.
        """
        pass


class DropMissing(MissingValueStrategy):
    """Drops rows containing missing values."""
    def __init__(self, subset: Optional[List[str]] = None, how: str = "any"):
        self.subset = subset
        self.how = how

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drops rows with NaN values according to the specified subset and how criteria."""
        return df.dropna(subset=self.subset, how=self.how).copy()


class FillMean(MissingValueStrategy):
    """Fills missing values in numerical columns with the respective column mean."""
    def __init__(self, columns: Optional[List[str]] = None):
        self.columns = columns

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        df_cleaned = df.copy()
        target_cols = (
            self.columns 
            if self.columns is not None 
            else df_cleaned.select_dtypes(include=[np.number]).columns
        )
        
        for col in target_cols:
            if col in df_cleaned.columns and df_cleaned[col].isnull().any():
                col_mean = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(col_mean)
        return df_cleaned


class FillMode(MissingValueStrategy):
    """Fills missing values in specified columns (or all columns) with the most frequent value."""
    def __init__(self, columns: Optional[List[str]] = None):
        self.columns = columns

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        df_cleaned = df.copy()
        target_cols = self.columns if self.columns is not None else df_cleaned.columns
        
        for col in target_cols:
            if col in df_cleaned.columns and df_cleaned[col].isnull().any():
                modes = df_cleaned[col].mode()
                if not modes.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(modes[0])
        return df_cleaned


class DataCleaner:
    """
    Context class that executes a chosen MissingValueStrategy.
    """
    def __init__(self, strategy: MissingValueStrategy):
        self._strategy: MissingValueStrategy = strategy

    @property
    def strategy(self) -> MissingValueStrategy:
        return self._strategy

    @strategy.setter
    def strategy(self, strategy: MissingValueStrategy) -> None:
        self._strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes the active strategy on the given DataFrame."""
        return self._strategy.handle(df)


### Verification — Exercise 2: Comparing Strategies

In [6]:
print("Original Raw Data:")
display(df_raw)

# Strategy A: Mean Imputation
cleaner = DataCleaner(FillMean())
df_mean_cleaned = cleaner.clean(df_raw)
print("\n--- Cleaned with FillMean Strategy ---")
display(df_mean_cleaned)

# Strategy B: Mode Imputation
cleaner.strategy = FillMode()
df_mode_cleaned = cleaner.clean(df_raw)
print("\n--- Cleaned with FillMode Strategy ---")
display(df_mode_cleaned)

# Strategy C: Row Dropping
cleaner.strategy = DropMissing()
df_drop_cleaned = cleaner.clean(df_raw)
print("\n--- Cleaned with DropMissing Strategy ---")
display(df_drop_cleaned)


Original Raw Data:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0



--- Cleaned with FillMean Strategy ---


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0



--- Cleaned with FillMode Strategy ---


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0



--- Cleaned with DropMissing Strategy ---


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


---
## Exercise 3: Add Decorators for Logging and Timing (Decorator Pattern)

### Objective
- Learn the **Decorator Pattern** in Python to add cross-cutting concerns (logging, timing) to methods transparently.
- Reinforce the **Open/Closed Principle**: extend class behavior without modifying source code.

### Concept
Decorators wrap functions or methods dynamically. In data engineering pipelines, tracking execution flow and measuring execution time are essential for debugging and performance optimization.

### Instructions
1. Implement two function decorators using `functools.wraps`:
   - `@log_action`: Logs when a method begins and finishes execution.
   - `@log_time`: Computes and prints elapsed execution time.
2. Apply these decorators to key methods in `CSVReader` and `DataCleaner`.


In [7]:
def log_action(func):
    """Decorator that logs when a function/method starts and completes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] >>> Starting method: '{func.__name__}'")
        result = func(*args, **kwargs)
        print(f"[LOG] <<< Finished method: '{func.__name__}'")
        return result
    return wrapper


def log_time(func):
    """Decorator that measures and prints execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_time = time.perf_counter() - start_time
        print(f"[TIME] Method '{func.__name__}' executed in {elapsed_time:.6f} seconds.")
        return result
    return wrapper


### Applying Decorators to `CSVReader` and `DataCleaner`

In [10]:
class LoggedCSVReader(CSVReader):
    """CSVReader subclass decorated with logging and timing."""
    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        return super().read()


class LoggedDataCleaner(DataCleaner):
    """DataCleaner subclass decorated with logging and timing."""
    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return super().clean(df)


### Verification — Exercise 3: Testing Decorated Classes

In [13]:
logged_reader = LoggedCSVReader(sample_file)
data = logged_reader.read()

logged_cleaner = LoggedDataCleaner(FillMean())
cleaned_data = logged_cleaner.clean(data)

[LOG] >>> Starting method: 'read'
[TIME] Method 'read' executed in 0.002353 seconds.
[LOG] <<< Finished method: 'read'
[LOG] >>> Starting method: 'clean'
[TIME] Method 'clean' executed in 0.001781 seconds.
[LOG] <<< Finished method: 'clean'


---
## Exercise 4: Implement Factory Pattern for Data Transformations

### Objective
- Master the **Factory Pattern** for scalable, decoupled creation of transformation objects.
- Combine **abstraction** (`DataTransform`) and **composition**.

### Concept
Instead of hardcoding concrete transformation classes across client scripts, a **Factory** encapsulates instantiation logic. Adding new transformations requires only adding a new subclass and registering it, keeping client code completely decoupled.

### Instructions
1. Create an abstract base class `DataTransform` with an abstract method `apply(df: pd.DataFrame) -> pd.DataFrame`.
2. Implement three transformation subclasses:
   - `NormalizeColumns`: Min-Max normalization for numeric columns $(x - x_{min}) / (x_{max} - x_{min})$.
   - `RemoveDuplicates`: Drops duplicate rows.
   - `StandardizeText`: Cleans text columns (whitespace stripping and case normalization).
3. Create a `TransformFactory` that returns the appropriate `DataTransform` object based on a string identifier.


In [14]:
class DataTransform(ABC):
    """Abstract base class for data transformation operations."""
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Applies transformation and returns transformed DataFrame."""
        pass


class NormalizeColumns(DataTransform):
    """Performs Min-Max normalization: (x - min) / (max - min) on numeric columns."""
    def __init__(self, columns: Optional[List[str]] = None):
        self.columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df_norm = df.copy()
        target_cols = (
            self.columns 
            if self.columns is not None 
            else df_norm.select_dtypes(include=[np.number]).columns
        )
        
        for col in target_cols:
            if col in df_norm.columns:
                c_min = df_norm[col].min()
                c_max = df_norm[col].max()
                if c_max > c_min:
                    df_norm[col] = (df_norm[col] - c_min) / (c_max - c_min)
                elif c_max == c_min:
                    df_norm[col] = 0.0
        return df_norm


class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the dataset."""
    def __init__(self, subset: Optional[List[str]] = None):
        self.subset = subset

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates(subset=self.subset).copy()


class StandardizeText(DataTransform):
    """Strips leading/trailing whitespace and standardizes case for string columns."""
    def __init__(self, columns: Optional[List[str]] = None, case: str = "title"):
        self.columns = columns
        self.case = case.lower()

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df_std = df.copy()
        target_cols = (
            self.columns 
            if self.columns is not None 
            else df_std.select_dtypes(include=["object"]).columns
        )
        
        for col in target_cols:
            if col in df_std.columns:
                series = df_std[col].astype(str).str.strip()
                if self.case == "lower":
                    df_std[col] = series.str.lower()
                elif self.case == "upper":
                    df_std[col] = series.str.upper()
                elif self.case == "title":
                    df_std[col] = series.str.title()
                else:
                    df_std[col] = series
        return df_std


class TransformFactory:
    """
    Factory class responsible for instantiating DataTransform objects.
    """
    _registry = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create_transform(cls, transform_type: str, **kwargs) -> DataTransform:
        """
        Factory method returning an instance of DataTransform.
        
        Args:
            transform_type (str): Key matching one of ['normalize', 'remove_duplicates', 'standardize_text'].
            **kwargs: Arguments forwarded to the concrete transform constructor.
            
        Returns:
            DataTransform: Instantiated transformation object.
        """
        key = transform_type.strip().lower()
        if key not in cls._registry:
            valid_keys = list(cls._registry.keys())
            raise ValueError(f"Unknown transform '{transform_type}'. Supported: {valid_keys}")
        
        return cls._registry[key](**kwargs)


### Verification — Exercise 4: Testing the Factory

In [15]:
# Create transformations using the factory
text_transformer = TransformFactory.create_transform("standardize_text", columns=["city"], case="title")
normalizer = TransformFactory.create_transform("normalize", columns=["age", "score"])
deduplicator = TransformFactory.create_transform("remove_duplicates")

# Apply sequence of transformations to mean-imputed data
df_transformed = text_transformer.apply(df_mean_cleaned)
df_transformed = normalizer.apply(df_transformed)
df_transformed = deduplicator.apply(df_transformed)

print("Transformed DataFrame (Text Standardized + Normalized):")
display(df_transformed)


Transformed DataFrame (Text Standardized + Normalized):


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,165.0,68.0,New York,0.588235
1,2,Bob,0.40625,172.0,70.0,Los Angeles,0.882353
2,3,Charlie,1.00000,168.0,72.0,Chicago,0.647059
3,4,David,0.40625,167.0,80.0,Houston,0.000000
4,5,Eva,0.00000,160.0,55.0,New York,0.764706
5,6,Frank,0.37500,175.0,85.0,Los Angeles,0.647059
6,7,Grace,0.40625,162.0,60.0,Chicago,1.000000


---
## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objective
- Combine all previous modules (`CSVReader`, `DataCleaner`, `DataTransform`, decorators) into a unified end-to-end data engineering pipeline.
- Apply the **Template Method Pattern** to establish an invariant execution skeleton while letting subclasses implement or customize individual pipeline phases.

### Concept
The **Template Method Pattern** defines the algorithm structure inside a base method (`run()`), calling primitive operations in a predefined sequence:
1. `load()`
2. `clean()`
3. `transform()`
4. `save()`

Subclasses override the individual steps without altering the orchestration workflow.

### Instructions
1. Create an abstract class `DataPipeline` with the template method `run()` invoking `load()`, `clean()`, `transform()`, and `save()`.
2. Implement `CSVDataPipeline` integrating `CSVReader`, `DataCleaner`, and a sequence of `DataTransform` objects created by `TransformFactory`.
3. Execute the complete pipeline on `sample_data (2).csv` and persist the output to `cleaned_sample_data.csv`.


In [17]:
class DataPipeline(ABC):
    """
    Abstract Base Class establishing the Template Method for data pipelines.
    """
    def run(self) -> pd.DataFrame:
        """
        Template method defining the invariant sequence of pipeline steps.
        """
        print("==================================================")
        print("          STARTING DATA CLEANING PIPELINE         ")
        print("==================================================")
        
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        
        print("==================================================")
        print("          PIPELINE EXECUTION COMPLETED            ")
        print("==================================================")
        return df

    @abstractmethod
    def load(self) -> pd.DataFrame:
        """Step 1: Load data from source."""
        pass

    @abstractmethod
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Step 2: Clean missing or corrupt data."""
        pass

    @abstractmethod
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Step 3: Apply feature engineering or transformations."""
        pass

    @abstractmethod
    def save(self, df: pd.DataFrame) -> None:
        """Step 4: Save cleaned dataset to destination."""
        pass


class CSVDataPipeline(DataPipeline):
    """
    Concrete pipeline implementation using CSVReader, DataCleaner, and DataTransforms.
    """
    def __init__(
        self,
        input_path: str,
        output_path: str,
        missing_strategy: MissingValueStrategy,
        transforms: Optional[List[DataTransform]] = None,
    ):
        self.input_path = input_path
        self.output_path = output_path
        self.reader = LoggedCSVReader(input_path)
        self.cleaner = LoggedDataCleaner(missing_strategy)
        self.transforms = transforms or []

    def load(self) -> pd.DataFrame:
        print(f"\n[Pipeline Step 1/4] Loading CSV data from '{self.input_path}'...")
        return self.reader.read()

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        print(f"\n[Pipeline Step 2/4] Handling missing values using {self.cleaner.strategy.__class__.__name__}...")
        return self.cleaner.clean(df)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        print(f"\n[Pipeline Step 3/4] Executing {len(self.transforms)} transformation steps...")
        result = df.copy()
        for idx, t in enumerate(self.transforms, start=1):
            print(f"  ({idx}/{len(self.transforms)}) Applying {t.__class__.__name__}...")
            result = t.apply(result)
        return result

    def save(self, df: pd.DataFrame) -> None:
        print(f"\n[Pipeline Step 4/4] Persisting output to '{self.output_path}'...")
        df.to_csv(self.output_path, index=False)
        print(f"[SUCCESS] Saved {len(df)} rows to '{self.output_path}'.")


### Verification — Exercise 5: End-to-End Pipeline Execution

In [18]:
# 1. Define transformation sequence via TransformFactory
pipeline_transforms = [
    TransformFactory.create_transform("standardize_text", columns=["city"], case="title"),
    TransformFactory.create_transform("normalize", columns=["age", "height_cm", "weight_kg", "score"]),
    TransformFactory.create_transform("remove_duplicates")
]

# 2. Configure and run the complete CSVDataPipeline
input_csv = "sample_data (2).csv"
output_csv = "cleaned_sample_data.csv"

pipeline = CSVDataPipeline(
    input_path=input_csv,
    output_path=output_csv,
    missing_strategy=FillMean(),
    transforms=pipeline_transforms
)

# 3. Execute pipeline run()
final_dataset = pipeline.run()

# 4. Preview the resulting cleaned dataset
print("\nFinal Cleaned & Normalized Dataset Preview:")
display(final_dataset)


          STARTING DATA CLEANING PIPELINE         

[Pipeline Step 1/4] Loading CSV data from 'sample_data (2).csv'...
[LOG] >>> Starting method: 'read'
[TIME] Method 'read' executed in 0.002433 seconds.
[LOG] <<< Finished method: 'read'

[Pipeline Step 2/4] Handling missing values using FillMean...
[LOG] >>> Starting method: 'clean'
[TIME] Method 'clean' executed in 0.000742 seconds.
[LOG] <<< Finished method: 'clean'

[Pipeline Step 3/4] Executing 3 transformation steps...
  (1/3) Applying StandardizeText...
  (2/3) Applying NormalizeColumns...
  (3/3) Applying RemoveDuplicates...

[Pipeline Step 4/4] Persisting output to 'cleaned_sample_data.csv'...
[SUCCESS] Saved 7 rows to 'cleaned_sample_data.csv'.
          PIPELINE EXECUTION COMPLETED            

Final Cleaned & Normalized Dataset Preview:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


### Verification — Reading Back the Saved CSV File

In [19]:
# Verify saved file exists and can be loaded with CSVReader
saved_reader = CSVReader(output_csv)
saved_df = saved_reader.read()
print(f"Read back {len(saved_df)} records from '{output_csv}'. No missing values remaining: {saved_df.isnull().sum().sum() == 0}")
display(saved_df)


Read back 7 records from 'cleaned_sample_data.csv'. No missing values remaining: True


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


---
## Summary of Applied OOP Principles & Design Patterns

| Pattern / Concept | Where Applied | Key Benefit |
| :--- | :--- | :--- |
| **Encapsulation & OOP** | `CSVReader` | Groups file loading, validation, and preview logic into a reusable object. |
| **Strategy Pattern** | `MissingValueStrategy`, `DataCleaner` | Makes missing-value handling algorithms (`DropMissing`, `FillMean`, `FillMode`) interchangeable without modifying cleaner code. |
| **Decorator Pattern** | `@log_action`, `@log_time` | Extends methods dynamically with timing and logging while adhering to the Open/Closed Principle. |
| **Factory Pattern** | `TransformFactory`, `DataTransform` | Decouples transformation instantiation from client code, making adding new transforms simple. |
| **Template Method Pattern** | `DataPipeline`, `CSVDataPipeline` | Preserves algorithm invariant steps (`load` -> `clean` -> `transform` -> `save`) while letting subclasses customize details. |
